In [33]:
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
import sys
from pathlib import Path


In [5]:
# подгружаем .env
load_dotenv()

True

In [6]:
# Считываем все креды
src_host = os.environ.get('DB_SOURCE_HOST')
src_port = os.environ.get('DB_SOURCE_PORT')
src_username = os.environ.get('DB_SOURCE_USER')
src_password = os.environ.get('DB_SOURCE_PASSWORD')
src_db = os.environ.get('DB_SOURCE_NAME') 

dst_host = os.environ.get('DB_DESTINATION_HOST')
dst_port = os.environ.get('DB_DESTINATION_PORT')
dst_username = os.environ.get('DB_DESTINATION_USER')
dst_password = os.environ.get('DB_DESTINATION_PASSWORD')
dst_db = os.environ.get('DB_DESTINATION_NAME')

s3_bucket = os.environ.get('S3_BUCKET_NAME')
s3_access_key = os.environ.get('AWS_ACCESS_KEY_ID')
s3_secret_access_key = os.environ.get('AWS_SECRET_ACCESS_KEY')

In [9]:

engine = create_engine(f'postgresql://{dst_username}:{dst_password}@{dst_host}:{dst_port}/{dst_db}')


In [11]:
SRC_TABLE = "public.real_estate_dataset_raw"
df = pd.read_sql(f"SELECT * FROM {SRC_TABLE}", con=engine)

df.head()


,flat_id,building_id,floor,kitchen_area,living_area,rooms,is_apartment,studio,total_area,price,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator
0,0,6220,9,9.9,19.900000,1,False,False,35.099998,9500000,1965,6,55.717113,37.781120,2.64,84,12,True
1,1,18012,7,0.0,16.600000,1,False,False,43.000000,13500000,2001,2,55.794849,37.608013,3.00,97,10,True
2,2,17821,9,9.0,32.000000,2,False,False,56.000000,13500000,2000,4,55.740040,37.761742,2.70,80,10,True
3,3,18579,1,10.1,43.099998,3,False,False,76.000000,20000000,2002,4,55.672016,37.570877,2.64,771,17,True
4,4,9293,3,3.0,14.000000,1,False,False,24.000000,5200000,1971,1,55.808807,37.707306,2.60,208,9,True


In [30]:
df.shape

(141362, 18)

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 141362 entries, 0 to 141361
Data columns (total 18 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   flat_id            141362 non-null  int64  
 1   building_id        141362 non-null  int64  
 2   floor              141362 non-null  int64  
 3   kitchen_area       141362 non-null  float64
 4   living_area        141362 non-null  float64
 5   rooms              141362 non-null  int64  
 6   is_apartment       141362 non-null  bool   
 7   studio             141362 non-null  bool   
 8   total_area         141362 non-null  float64
 9   price              141362 non-null  int64  
 10  build_year         141362 non-null  int64  
 11  building_type_int  141362 non-null  int64  
 12  latitude           141362 non-null  float64
 13  longitude          141362 non-null  float64
 14  ceiling_height     141362 non-null  float64
 15  flats_count        141362 non-null  int64  
 16  fl

In [13]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
flat_id,141362.0,NaN,NaN,NaN,70680.5,40807.838714,0.0,35340.25,70680.5,106020.75,141361.0
building_id,141362.0,NaN,NaN,NaN,14053.665235,6988.831066,1.0,8535.25,14332.0,20475.0,24620.0
floor,141362.0,NaN,NaN,NaN,7.467346,5.717144,1.0,3.0,6.0,10.0,56.0
kitchen_area,141362.0,NaN,NaN,NaN,9.001579,5.264076,0.0,6.1,8.8,10.2,203.0
living_area,141362.0,NaN,NaN,NaN,31.056948,23.96864,0.0,19.0,29.4,41.400002,700.0
rooms,141362.0,NaN,NaN,NaN,2.129476,0.99434,1.0,1.0,2.0,3.0,20.0
is_apartment,141362,2,False,139990,NaN,NaN,NaN,NaN,NaN,NaN,NaN
studio,141362,1,False,141362,NaN,NaN,NaN,NaN,NaN,NaN,NaN
total_area,141362.0,NaN,NaN,NaN,62.374644,40.295864,11.0,39.299999,53.0,72.0,960.299988
price,141362.0,NaN,NaN,NaN,19441619.951904,66269544.299228,11.0,8900000.0,11850000.0,16950000.0,9873737728.0


Проверка дубликатов

In [23]:
dup_all = df.duplicated().sum()
dup_flat_id = df.duplicated(subset=["flat_id"]).sum() if "flat_id" in df.columns else None
dup_all, dup_flat_id

(0, 0)

проверка пропусков

In [ ]:
na_counts = df.isna().sum().sort_values(ascending=False)

na_counts

,missing_cnt,missing_%


Проверка нулевых/негативных значений

In [29]:
checks = {}

for col in ["price", "total_area", "kitchen_area", "living_area"]:
    if col in df.columns:
        checks[f"{col}_nonpositive"] = int((df[col] <= 0).sum())

if "rooms" in df.columns:
    checks["rooms_nonpositive"] = int((df["rooms"] <= 0).sum())

checks


{'price_nonpositive': 0,
 'total_area_nonpositive': 0,
 'kitchen_area_nonpositive': 11701,
 'living_area_nonpositive': 18588,
 'rooms_nonpositive': 0}

Проверка выбросов

In [31]:
def iqr_bounds(s: pd.Series, k: float = 1.5):
    s = s.dropna()
    q1, q3 = np.percentile(s, [25, 75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

outlier_report = []
for col in ["price", "total_area", "kitchen_area", "living_area"]:
    if col in df.columns:
        lo, hi = iqr_bounds(df[col])
        n_out = int(((df[col] < lo) | (df[col] > hi)).sum())
        outlier_report.append((col, lo, hi, n_out, round(100*n_out/len(df), 2)))

pd.DataFrame(outlier_report, columns=["feature", "iqr_lo", "iqr_hi", "outliers_cnt", "outliers_%"])


,feature,iqr_lo,iqr_hi,outliers_cnt,outliers_%
0,price,-3.175000e+06,2.902500e+07,14318,10.13
1,total_area,-9.750002e+00,1.210500e+02,7839,5.55
2,kitchen_area,-4.999995e-02,1.635000e+01,8076,5.71
3,living_area,-1.460000e+01,7.500000e+01,4962,3.51


In [34]:
sys.path.append(str(Path("..").resolve()))
sys.path.append(str(Path("../plugins").resolve()))

from cleaning_utils import clean_dataset

Пример очистки

In [ ]:
df_clean = clean_dataset(df)

print("raw:", df.shape)
print("clean:", df_clean.shape)

# дубликаты после
if "flat_id" in df_clean.columns:
    print("dup flat_id after:", df_clean.duplicated(subset=["flat_id"]).sum())

# пропуски после
(df_clean.isna().sum().sort_values(ascending=False).head(15))



raw: (141362, 18)
clean: (119522, 18)
dup flat_id after: 0


flat_id              0
building_id          0
floor                0
kitchen_area         0
living_area          0
rooms                0
is_apartment         0
studio               0
total_area           0
price                0
build_year           0
building_type_int    0
latitude             0
longitude            0
ceiling_height       0
dtype: int64

В итоге поскольку в нашем датасете нет пропусков и дубликатов я только удаляю значения где kitchen area и living area равны 0.